# Part 2.5 Memory

After this notebook you can give an agent memory the team writes and memory the agent
writes itself, and show one thread using what another thread learned.

In [ ]:
from kea.config import MODEL, setup
from kea.notebook import ask_and_approve, final, steps

setup("kea-rebooking-local")

## See it

A passenger tells the agent a lasting preference in one thread. A second thread, with
nothing carried over in its messages, uses it.

<img src="../../assets/screenshots/p2-05-memory-trace.png" alt="the trace of thread alex-2: ls /memories/, readfile /memories/preferences.md, then rebook(..., seatpreference='aisle')." width="900">

## Build it

Deep Agents has two kinds of memory, and the difference is who writes them.

- `AGENTS.md` is a file the team writes. It is loaded into the system prompt before every
  model call.
- `/memories/` holds what the agent learns. It writes files there itself, and a later
  conversation reads them.

Here is the team's file:

In [ ]:
from kea.rebooking.agent import HERE

print((HERE / "memory" / "AGENTS.md").read_text())

Steps 4 and 5 mention `policy-checker` and `issue_credit`. Those arrive in
[2.7](07_subagents.ipynb) and [2.8](08_middleware.ipynb). This agent doesn't have them
yet. The `/memory/` route and the `/memories/` route come from the backend in
[2.4](04_backends.ipynb); we import it.

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore

from kea.rebooking.agent import build_backend
from kea.rebooking.tools import get_booking, rebook, search_flights

PREFERENCES = ("kea-air", "passenger-preferences", "local")
store = InMemoryStore()

memory_agent = create_deep_agent(
    model=MODEL,
    tools=[get_booking, search_flights, rebook],
    system_prompt="You are Kea Air's rebooking assistant.",
    memory=["/memory/AGENTS.md"],
    backend=build_backend(),
    store=store,
    checkpointer=InMemorySaver(),
)

await ask_and_approve(
    memory_agent,
    "KA 1182 got cancelled. Confirmation ABC123, last name Example. Please rebook me, "
    "and remember that I always want an aisle seat.",
    thread_id="alex-1",
)
print([(i.key, i.value["content"]) for i in store.search(PREFERENCES)])

In [ ]:
result = await ask_and_approve(
    memory_agent,
    "Hi, Alex Example again, confirmation ABC123. Please rebook me.",
    thread_id="alex-2",
)
steps(result)
print()
final(result)

## Under the hood

<img src="../../assets/slides/langgraph-memory-across-threads.jpg" width="720">

The checkpointer holds one thread's messages (short-term). The Store holds what outlives
threads (long-term). `/memories/` is a Store route, so the agent's file tools reach it.

- **Recall is retrieval, not the model remembering.** In the second thread the agent
  called `ls` and `read_file` on `/memories/`, and the preference entered the conversation
  as a tool result. When memory misbehaves, check three things: what was written, what
  was retrieved, what reached the prompt.
- **Whose words win.** The booking says `window` and the memory says `aisle`. `AGENTS.md`
  tells the agent to apply what it finds in `/memories/`, so the passenger's own words
  win.
- **Memory the agent writes needs care.** Tell it what qualifies so it doesn't hoard, tell
  it to update the file so it doesn't contradict itself, and scope the namespace per
  passenger.

Docs: [Deep Agents memory](https://docs.langchain.com/oss/python/deepagents/memory).

## Recap

- `AGENTS.md` is memory the team writes. It is always in the prompt.
- `/memories/` is memory the agent writes. A Store route keeps it across threads.
- Debug memory by checking what was written, what was retrieved, and what reached the
  prompt.

**Next:** [Part 2.6 Skills](06_skills.ipynb)